In [ ]:
# 1. Imports + global configuration   [MODIFIED: original cells 0 and 0b merged]
# Serves: seeds, shared training recipe, release-readiness.
import os, sys, glob, time, json, gc, math, random, platform, itertools, warnings, shutil, subprocess, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy import stats
from scipy.stats import beta, binomtest, rankdata
warnings.filterwarnings("ignore")

# ------------------------------ paths ------------------------------
IS_KAGGLE = os.path.exists("/kaggle")
BASE = "/kaggle/working" if IS_KAGGLE else "/content"
DATA_ROOT = os.path.join(BASE, "iccad12_dataset", "iccad-official")
OUT_DIR = os.path.join(BASE, "outputs")
RAW_DIR = os.path.join(OUT_DIR, "raw")      # per-run scores (.npz) + metric rows (.json); analysis cells read ONLY this
TAB_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
CACHE_DIR = "/kaggle/temp/litho_cache" if os.path.exists("/kaggle/temp") else os.path.join(BASE, "cache")
for _d in (OUT_DIR, RAW_DIR, TAB_DIR, FIG_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

# ------------------------- experiment design -------------------------
IMG_SIZE = 96
ALL_BENCHMARKS = [1, 2, 3, 4, 5]
BENCHMARKS = list(ALL_BENCHMARKS)         # cell 14 narrows this to the benchmarks that actually have finished runs
SEEDS = [0, 1, 2, 3, 4]                    # the seed controls: 85/15 split, weight init, augmentation, shuffling, RF/DT random_state
QUICK_PILOT = False                         # True -> only the first seed; False -> RUN_SEEDS = SEEDS (or set RUN_SEEDS by hand)
RUN_SEEDS = SEEDS[:1] if QUICK_PILOT else SEEDS

BACKBONES = ["ref", "proposed"]            # ref = reimplemented reference-paper CNN (6,949 params); proposed = residual depthwise-separable CNN
LOSSES = ["focal", "bce"]                  # focal (gamma=2, alpha=.75) vs class-weighted binary cross-entropy
HEADS = ["SVM", "RF", "LR", "DT"]          # classical heads (Decision Tree added)
MAIN_CONFIG = ("proposed", "focal")        # the DA-2 headline configuration
FULL_TAP_COMBOS = {MAIN_CONFIG}            # early/late/combined taps only here; other configs use the late tap only (saves time)
RUN_PARAM_MATCHED = False                  # True -> also train a widened reference CNN with ~the proposed model's parameter count

# ONE training recipe shared by BOTH CNNs (this is what "common baseline: same parameters and loss" requires)
TRAIN_CFG = dict(optimizer="nadam", lr=1e-3, epochs=20, patience=5,
                 reduce_lr_factor=0.5, reduce_lr_patience=2, min_lr=1e-6,
                 batch_size=32, augment=True, val_fraction=0.15,
                 min_steps=100, es_start_epoch=3)   # min_steps: floor on optimizer steps per epoch (tiny benchmarks); es_start_epoch: early stopping ignores epochs < 3
FOCAL = dict(gamma=2.0, alpha=0.75)
CLASS_WEIGHT_CAP = 10.0                    # cap on the positive-class weight used by the weighted-BCE loss
# Proposed-backbone details. Defaults reproduce the DA-2 CODE (relu, SpatialDropout 0.1); the DA-2 paper text said ELU / 0.2.
# Either keep these and fix the paper text, or change them here (and rerun).
PROPOSED_ARCH = dict(act="relu", spatial_dropout=0.1, dense_dropout=0.5, l2=1e-4)

SVM_C_GRID = [0.5, 1.0, 5.0]
DT_DEPTHS = [3, 5, 8, None]
DT_LEAVES = [1, 5, 20]
RF_TREES = 300
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# 2. Reproducibility + environment report   [NEW]
# Serves: "release-ready description of code and seeds".
DETERMINISTIC = False   # True = tf op determinism (slower; some GPU ops may raise). Seed-to-seed variance is what we REPORT, so default off.

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        pass

if DETERMINISTIC:
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception as e:
        print("Could not enable op determinism:", e)

def environment_report():
    import sklearn, scipy
    gpus = tf.config.list_physical_devices("GPU")
    gpu_names = []
    for g in gpus:
        try:
            gpu_names.append(tf.config.experimental.get_device_details(g).get("device_name", "GPU"))
        except Exception:
            gpu_names.append("GPU")
    return dict(python=platform.python_version(), platform=platform.platform(), tensorflow=tf.__version__,
                numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, scipy=scipy.__version__,
                gpus=gpu_names, n_cpu=os.cpu_count(), deterministic=DETERMINISTIC)

ENV = environment_report()
CONFIG_SNAPSHOT = dict(IMG_SIZE=IMG_SIZE, BENCHMARKS=BENCHMARKS, SEEDS=SEEDS, RUN_SEEDS=RUN_SEEDS, BACKBONES=BACKBONES,
                       LOSSES=LOSSES, HEADS=HEADS, MAIN_CONFIG=list(MAIN_CONFIG), TRAIN_CFG=TRAIN_CFG, FOCAL=FOCAL,
                       CLASS_WEIGHT_CAP=CLASS_WEIGHT_CAP, PROPOSED_ARCH=PROPOSED_ARCH, SVM_C_GRID=SVM_C_GRID,
                       DT_DEPTHS=[str(d) for d in DT_DEPTHS], DT_LEAVES=DT_LEAVES, RF_TREES=RF_TREES)
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(dict(config=CONFIG_SNAPSHOT, environment=ENV), f, indent=2)
print(json.dumps(ENV, indent=2))
if not ENV["gpus"]:
    print("\nWARNING: no GPU detected. Latency numbers will be CPU numbers and training will be slow "
          "(the DA-2 Kaggle run used accelerator=none and took ~3.1 h for ONE seed).")

In [ ]:
# 3. Download + extract the dataset (skipped if already extracted)   [MODIFIED: original cells 2-4 merged]
# Serves: release-readiness (idempotent, records the archive checksum).
FILE_ID = "1jx7gDR92sqoIw2Nh4NGwwZwC-qNp2osd"
SHARE_URL = "https://drive.google.com/file/d/1jx7gDR92sqoIw2Nh4NGwwZwC-qNp2osd/view?usp=share_link"
ARCHIVE = os.path.join(BASE, "iccad12_dataset.zip")      # despite the name this is a RAR archive
EXTRACT_DIR = os.path.join(BASE, "iccad12_dataset")

def _is_rar(path):
    try:
        with open(path, "rb") as f:
            return f.read(4) == b"Rar!"
    except OSError:
        return False

if not os.path.isdir(DATA_ROOT):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gdown", "rarfile"], check=False)
    subprocess.run("apt-get -qq install -y unrar > /dev/null 2>&1", shell=True)
    import gdown, rarfile
    gdown.download(id=FILE_ID, output=ARCHIVE, quiet=False)
    if not _is_rar(ARCHIVE):                              # first attempt sometimes returns an HTML page
        if os.path.exists(ARCHIVE):
            os.remove(ARCHIVE)
        gdown.download(url=SHARE_URL, output=ARCHIVE, quiet=False, fuzzy=True)
    assert _is_rar(ARCHIVE), "Download did not return a RAR archive. Check Internet access / the Drive link."
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with rarfile.RarFile(ARCHIVE) as rf:
        rf.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Dataset already extracted at", DATA_ROOT)

DATASET_SHA256 = None
if os.path.exists(ARCHIVE):
    h = hashlib.sha256()
    with open(ARCHIVE, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    DATASET_SHA256 = h.hexdigest()
print("Archive SHA-256:", DATASET_SHA256)
assert os.path.isdir(DATA_ROOT), f"Expected folder not found: {DATA_ROOT}"

In [ ]:
# 4. Folder-structure sanity check   [KEPT]
def print_tree(root, max_depth=3, max_items=6):
    root = os.path.abspath(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(dirpath) or dirpath}/")
        sub_indent = "  " * (depth + 1)
        for f in filenames[:max_items]:
            print(f"{sub_indent}{f}")
        if len(filenames) > max_items:
            print(f"{sub_indent}... ({len(filenames)} files total)")

print_tree(DATA_ROOT)

In [ ]:
# 5. File listing, class distribution (Table I / Fig. 1) and validation-set sizes   [MODIFIED]
# Serves: honest reporting of how few validation hotspots there are (drives threshold-selection noise, Sec. VII-B).
def list_files_labels(benchmark, split):
    """split is 'train' or 'test'. Returns (filepaths, labels) with label 1 = HS, 0 = NHS."""
    base = os.path.join(DATA_ROOT, f"iccad{benchmark}", split)
    hs_files = sorted(glob.glob(os.path.join(base, f"{split}_hs", "*.png")))
    nhs_files = sorted(glob.glob(os.path.join(base, f"{split}_nhs", "*.png")))
    return hs_files + nhs_files, [1] * len(hs_files) + [0] * len(nhs_files)

_rows = []
for b in BENCHMARKS:
    _, tl = list_files_labels(b, "train")
    _, el = list_files_labels(b, "test")
    tl, el = np.array(tl), np.array(el)
    _, vl = train_test_split(tl, test_size=TRAIN_CFG["val_fraction"], stratify=tl, random_state=0)
    _rows.append(dict(Bench=f"ICCAD-{b}", train_HS=int(tl.sum()), train_NHS=int((1 - tl).sum()),
                      test_HS=int(el.sum()), test_NHS=int((1 - el).sum()),
                      ratio_train=f"1:{(1 - tl).sum() / max(tl.sum(), 1):.1f}",
                      ratio_test=f"1:{(1 - el).sum() / max(el.sum(), 1):.1f}",
                      val_n=len(vl), val_HS=int(vl.sum())))
DATA_SUMMARY = pd.DataFrame(_rows)
print(DATA_SUMMARY.to_string(index=False))
DATA_SUMMARY.to_csv(os.path.join(TAB_DIR, "table1_class_distribution.csv"), index=False)
print("\nNOTE: the validation split holds only", DATA_SUMMARY.val_HS.tolist(), "hotspots per benchmark: "
      "thresholds / C / depth are tuned on these, so expect noisy selection on ICCAD-1/4/5. Report this in the paper.")

fig, ax = plt.subplots(figsize=(7, 3.5))
w = 0.2; xs = np.arange(len(DATA_SUMMARY))
for i, (col, lab) in enumerate([("train_HS", "Train HS"), ("train_NHS", "Train NHS"), ("test_HS", "Test HS"), ("test_NHS", "Test NHS")]):
    ax.bar(xs + (i - 1.5) * w, DATA_SUMMARY[col], w, label=lab)
ax.set_yscale("log"); ax.set_xticks(xs); ax.set_xticklabels(DATA_SUMMARY.Bench); ax.set_ylabel("Count (log)"); ax.legend(fontsize=7)
ax.set_title("Class distribution per benchmark"); fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig1_class_distribution.png"), dpi=200); plt.show()

In [ ]:
# 6. Cached image arrays + tf.data / batched-prediction helpers   [MODIFIED: replaces the PNG-decoding tf.data pipeline]
# Serves: fair latency numbers (no disk/PNG cost inside any timed region) and much faster training.
def _load_u8(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_png(img, channels=1)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)

def load_images(files, batch=512):
    ds = tf.data.Dataset.from_tensor_slices(files).map(_load_u8, num_parallel_calls=AUTOTUNE).batch(batch).prefetch(AUTOTUNE)
    return np.concatenate([b.numpy() for b in ds], axis=0)

_ARR = {}
def get_split_arrays(b, split):
    """uint8 images (N,H,W,1) and int labels for one benchmark split, decoded once and cached on disk."""
    key = (b, split)
    if key in _ARR:
        return _ARR[key]
    path = os.path.join(CACHE_DIR, f"b{b}_{split}_{IMG_SIZE}.npz")
    if os.path.exists(path):
        z = np.load(path); X, y = z["X"], z["y"]
    else:
        files, labels = list_files_labels(b, split)
        X, y = load_images(files), np.asarray(labels, np.int32)
        np.savez(path, X=X, y=y)
    _ARR[key] = (X, y)
    return X, y

def release_arrays():
    _ARR.clear(); gc.collect()

def get_split(b, seed):
    """Seeded stratified 85/15 train/validation split of the official training set + the untouched official test set."""
    Xall, yall = get_split_arrays(b, "train")
    Xte, yte = get_split_arrays(b, "test")
    itr, iva = train_test_split(np.arange(len(yall)), test_size=TRAIN_CFG["val_fraction"], stratify=yall, random_state=seed)
    return Xall[itr], yall[itr], Xall[iva], yall[iva], Xte, yte

def make_ds(X, y, batch_size, shuffle=False, augment=False, seed=0):
    ds = tf.data.Dataset.from_tensor_slices((X, y.astype("float32")))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(X), 10000), seed=seed, reshuffle_each_iteration=True)
    def _prep(img, lab):
        img = tf.cast(img, tf.float32) / 255.0
        if augment:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_flip_up_down(img)
        return img, lab
    return ds.map(_prep, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)

def predict_batched(model, X, bs=512):
    outs = []
    for i in range(0, len(X), bs):
        xb = tf.convert_to_tensor(X[i:i + bs].astype(np.float32) / 255.0)
        outs.append(model(xb, training=False).numpy())
    return np.concatenate(outs, axis=0)

In [ ]:
# 7. Metrics + method-id helpers   [MODIFIED: adds FA count, explicit TP/FN/FP/TN, AUC/AP]
# Serves: published-results comparison (recall + false alarms), CIs, tests.
def confusion(y_true, y_pred):
    y = np.asarray(y_true).astype(bool); p = np.asarray(y_pred).astype(bool)
    return int((y & p).sum()), int((y & ~p).sum()), int((~y & p).sum()), int((~y & ~p).sum())   # tp, fn, fp, tn

def metrics_from_counts(tp, fn, fp, tn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return dict(precision=precision, recall=recall, specificity=specificity, f1=f1,
                balanced_accuracy=(recall + specificity) / 2)

def compute_metrics(y_true, y_pred):
    tp, fn, fp, tn = confusion(y_true, y_pred)
    out = metrics_from_counts(tp, fn, fp, tn)
    out.update(dict(confusion_matrix=[[tp, fn], [fp, tn]], tp=tp, fn=fn, fp=fp, tn=tn, fa=fp))   # fa = false-alarm count
    return out

def auc_metrics(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    try:
        return dict(roc_auc=float(roc_auc_score(y_true, y_score)), average_precision=float(average_precision_score(y_true, y_score)))
    except ValueError:
        return dict(roc_auc=float("nan"), average_precision=float("nan"))

# method id = "backbone|loss|head|tap"   (tap is "-" for end-to-end)
def mid(backbone, loss, head, tap):
    return "|".join([backbone, loss, head, tap])
def parse_mid(m):
    return m.split("|")
def skey(m):                       # key used inside .npz files
    return m.replace("|", "_")

BB_NAME = {"ref": "Reference", "proposed": "Proposed", "ref_wide": "Ref-wide", "rand_init": "RandInit", "vgg16": "VGG16"}
def pretty(m):
    bb, ls, h, tap = parse_mid(m)
    lstr = "" if ls == "none" else f", {ls}"
    if h == "e2e":
        return f"{BB_NAME.get(bb, bb)} CNN (end-to-end{lstr})"
    t = {"early": "E", "late": "L", "combined": "C"}.get(tap, tap)
    name = "Ensemble" if h == "ENS" else h
    return f"{name} ({t}) [{BB_NAME.get(bb, bb)}{lstr}]"

In [ ]:
# 8. Threshold selection (validation only) and operating points   [MODIFIED: find_best_threshold rewritten]
# Fixes vs DA-2: (a) ties -> MIDDLE of the optimal plateau (the old strict ">" always took the LOWEST tied threshold, which
# biases toward high recall / low specificity); (b) finer grid (0.1th-99.9th percentile, 400 points) instead of 1st-99th (99);
# (c) vectorised, and exposes the whole BA-vs-threshold curve for the sensitivity analysis.
# Serves: threshold sensitivity, validation-only selection. Rule: score >= threshold  => hotspot.
def ba_curve(y_true, y_score, thresholds):
    y = np.asarray(y_true).astype(int); s = np.asarray(y_score, dtype=np.float64)
    pos, neg = np.sort(s[y == 1]), np.sort(s[y == 0])
    n_pos, n_neg = len(pos), len(neg)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    tp = n_pos - np.searchsorted(pos, thresholds, side="left")
    fp = n_neg - np.searchsorted(neg, thresholds, side="left")
    tpr = tp / max(n_pos, 1); tnr = 1.0 - fp / max(n_neg, 1)
    return tpr, tnr, (tpr + tnr) / 2.0

def candidate_thresholds(y_score, n=400, lo=0.001, hi=0.999):
    return np.unique(np.quantile(np.asarray(y_score, dtype=np.float64), np.linspace(lo, hi, n)))

def find_best_threshold(y_true, y_score, return_curve=False):
    thr = candidate_thresholds(y_score)
    tpr, tnr, ba = ba_curve(y_true, y_score, thr)
    best = ba.max()
    tied = np.flatnonzero(ba >= best - 1e-12)
    t = float(thr[tied[len(tied) // 2]])          # middle of the tied-optimal set
    if return_curve:
        return t, float(best), (thr, tpr, tnr, ba)
    return t, float(best)

def operating_threshold(kind, y_true, y_score, target):
    """Alternative operating points chosen on VALIDATION data: kind='recall' (smallest-FA threshold with recall >= target)
    or kind='fpr' (highest-recall threshold with false-positive rate <= target)."""
    thr = candidate_thresholds(y_score)
    tpr, tnr, _ = ba_curve(y_true, y_score, thr)
    if kind == "recall":
        ok = thr[tpr >= target]
        return float(ok.max()) if len(ok) else float(thr.min())
    ok = thr[(1.0 - tnr) <= target]
    return float(ok.min()) if len(ok) else float(thr.max())

In [ ]:
# 9. Losses, the two backbones, and ONE shared training recipe   [MODIFIED: original cells 8 + 9]
# Serves: "common baseline (same parameters and loss)" and "separate backbone / loss effects".
#  * Both CNNs now share optimizer, lr, epochs, patience, ReduceLROnPlateau, batch size, augmentation (TRAIN_CFG).
#  * `loss` is a factor: "focal" or class-weighted "bce".
#  * Each backbone exposes an EARLY and a LATE tap so the same heads can run on either backbone.
class BalancedAccuracy(tf.keras.metrics.Metric):
    """Balanced accuracy at a fixed threshold; EarlyStopping monitors this so class collapse is not rewarded."""
    def __init__(self, name="balanced_accuracy", threshold=0.5, **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.tp = self.add_weight(name="tp", initializer="zeros")
        self.tn = self.add_weight(name="tn", initializer="zeros")
        self.fp = self.add_weight(name="fp", initializer="zeros")
        self.fn = self.add_weight(name="fn", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
        y_pred = tf.cast(tf.reshape(y_pred, [-1]) >= self.threshold, tf.float32)
        self.tp.assign_add(tf.reduce_sum(y_true * y_pred))
        self.tn.assign_add(tf.reduce_sum((1 - y_true) * (1 - y_pred)))
        self.fp.assign_add(tf.reduce_sum((1 - y_true) * y_pred))
        self.fn.assign_add(tf.reduce_sum(y_true * (1 - y_pred)))

    def result(self):
        eps = tf.keras.backend.epsilon()
        return (self.tp / (self.tp + self.fn + eps) + self.tn / (self.tn + self.fp + eps)) / 2.0

    def reset_state(self):
        for v in [self.tp, self.tn, self.fp, self.fn]:
            v.assign(0.0)


def focal_loss(gamma=2.0, alpha=0.75):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
        y_pred = tf.clip_by_value(tf.reshape(y_pred, [-1]), 1e-7, 1 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1.0), y_pred, 1 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1.0), alpha, 1 - alpha)
        return tf.reduce_mean(-alpha_t * tf.pow(1 - pt, gamma) * tf.math.log(pt))
    return loss_fn


def weighted_bce(w_neg, w_pos):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
        y_pred = tf.clip_by_value(tf.reshape(y_pred, [-1]), 1e-7, 1 - 1e-7)
        w = y_true * w_pos + (1 - y_true) * w_neg
        return tf.reduce_mean(-w * (y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred)))
    return loss_fn


def class_weights_from(y):
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=np.asarray(y))
    return float(cw[0]), float(min(cw[1], CLASS_WEIGHT_CAP))


def make_loss(name, y_train):
    if name == "focal":
        return focal_loss(**FOCAL)
    if name == "bce":
        w0, w1 = class_weights_from(y_train)
        return weighted_bce(w0, w1)
    raise ValueError(name)


def make_optimizer():
    if TRAIN_CFG["optimizer"] == "nadam":
        return tf.keras.optimizers.Nadam(learning_rate=TRAIN_CFG["lr"])
    return tf.keras.optimizers.Adam(learning_rate=TRAIN_CFG["lr"])


def compile_model(model, loss_fn):
    model.compile(optimizer=make_optimizer(), loss=loss_fn,
                  metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
                           tf.keras.metrics.Recall(name="recall"), BalancedAccuracy(),
                           tf.keras.metrics.AUC(name="auc")])


def basic_block(x, filters, name_prefix):
    """Reference paper's basic block: 3x Conv2D(3x3) -> BatchNorm -> ELU -> MaxPool(2x2)."""
    x = layers.Conv2D(filters, 3, padding="same", activation="elu", name=f"{name_prefix}_conv1")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="elu", name=f"{name_prefix}_conv2")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation=None, name=f"{name_prefix}_conv3")(x)
    x = layers.BatchNormalization(momentum=0.99, epsilon=0.001, name=f"{name_prefix}_bn")(x)
    x = layers.Activation("elu", name=f"{name_prefix}_act")(x)
    return layers.MaxPooling2D(pool_size=(2, 2), name=f"{name_prefix}_pool")(x)


def build_ref(loss_fn, filters=12, img_size=IMG_SIZE):
    """Reference-paper lightweight CNN (6,949 params at filters=12). Taps: early = GAP after block 1; late = Flatten."""
    inputs = layers.Input(shape=(img_size, img_size, 1))
    x = basic_block(inputs, filters, "base_bb1")
    early = layers.GlobalAveragePooling2D(name="early_features")(x)
    x = layers.MaxPooling2D(pool_size=(5, 5), name="base_extra_pool")(x)
    x = basic_block(x, filters, "base_bb2")
    late = layers.Flatten(name="late_features")(x)
    x = layers.Dropout(0.3)(late)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)
    model = models.Model(inputs, out, name="reference_baseline_cnn")
    compile_model(model, loss_fn)
    return model, {"early": models.Model(inputs, early), "late": models.Model(inputs, late)}


def residual_block(x, filters, name_prefix, a=PROPOSED_ARCH):
    reg = tf.keras.regularizers.l2(a["l2"])
    shortcut = x
    x = layers.SeparableConv2D(filters, 3, padding="same", depthwise_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(a["act"])(x)
    x = layers.SeparableConv2D(filters, 3, padding="same", depthwise_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding="same")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add(name=f"{name_prefix}_add")([x, shortcut])
    x = layers.Activation(a["act"])(x)
    return layers.SpatialDropout2D(a["spatial_dropout"])(x)


def build_proposed(loss_fn, img_size=IMG_SIZE, a=PROPOSED_ARCH):
    """Residual depthwise-separable backbone (163,489 params in DA-2). Taps: early = GAP after block 1 (64-d); late = Dense(128)."""
    inputs = layers.Input(shape=(img_size, img_size, 1))
    x = layers.Conv2D(32, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(a["act"])(x)
    x = layers.MaxPooling2D(2)(x)
    x = residual_block(x, 64, "res1")
    x = layers.MaxPooling2D(2)(x)
    early = layers.GlobalAveragePooling2D(name="early_features")(x)
    x = residual_block(x, 128, "res2")
    x = layers.MaxPooling2D(2)(x)
    x = residual_block(x, 192, "res3")
    x = layers.GlobalAveragePooling2D()(x)
    late = layers.Dense(128, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(a["l2"]), name="late_features")(x)
    x = layers.Dropout(a["dense_dropout"])(late)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)
    model = models.Model(inputs, out, name="proposed_residual_cnn")
    compile_model(model, loss_fn)
    return model, {"early": models.Model(inputs, early), "late": models.Model(inputs, late)}


_MATCHED = {}
def matched_filters():
    """Reference-CNN width whose parameter count is closest to the proposed backbone (parameter-matched control)."""
    if "f" not in _MATCHED:
        target = build_proposed(focal_loss())[0].count_params()
        best = None
        for f in range(12, 97, 2):
            n = build_ref(focal_loss(), filters=f)[0].count_params()
            if best is None or abs(n - target) < abs(best[1] - target):
                best = (f, n)
        tf.keras.backend.clear_session()
        _MATCHED["f"] = best[0]
        print(f"Parameter-matched reference CNN: filters={best[0]} -> {best[1]:,} params (target {target:,})")
    return _MATCHED["f"]


def build_backbone(name, loss_name, y_train):
    loss_fn = make_loss(loss_name, y_train)
    if name == "ref":
        return build_ref(loss_fn)
    if name == "ref_wide":
        return build_ref(loss_fn, filters=matched_filters())
    if name == "proposed":
        return build_proposed(loss_fn)
    raise ValueError(name)


def train_cnn(model, train_ds, val_ds):
    """Identical for every backbone / loss. Early stopping on validation ROC-AUC (threshold-free, so focal-loss probability
    scaling does not matter) and at least TRAIN_CFG['min_steps'] optimizer steps per epoch so tiny benchmarks (ICCAD-1) get
    enough updates for BatchNorm statistics to settle."""
    nb = int(tf.data.experimental.cardinality(train_ds).numpy())
    spe = max(nb, TRAIN_CFG["min_steps"]) if nb > 0 else None
    if spe is not None and spe > nb:
        ds, steps = train_ds.repeat(), spe        # reshuffles each pass; 'epoch' = spe batches
    else:
        ds, steps = train_ds, None
    cbs = [tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=TRAIN_CFG["patience"],
                                            start_from_epoch=TRAIN_CFG["es_start_epoch"], restore_best_weights=True),
           tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=TRAIN_CFG["reduce_lr_factor"],
                                                patience=TRAIN_CFG["reduce_lr_patience"], min_lr=TRAIN_CFG["min_lr"])]
    t0 = time.time()
    hist = model.fit(ds, validation_data=val_ds, epochs=TRAIN_CFG["epochs"], steps_per_epoch=steps,
                     callbacks=cbs, verbose=0)
    return time.time() - t0, len(hist.history["loss"])

In [ ]:
# 10. Classical heads, ensemble, and result persistence   [MODIFIED: adds Decision Tree, clipped ensemble, raw-score saving]
# Serves: Decision Tree + LR requirement, raw scores for CIs / tests / calibration, validation-only selection.
# Hyper-parameters (SVM C, DT depth/leaf) are chosen by VALIDATION ROC-AUC (threshold-free, so less noisy than BA on ~4-15 val hotspots);
# the decision threshold is then chosen on validation BA (cell 8). The test set is only scored, never used for selection.
def _auc(y, s):
    try:
        return float(roc_auc_score(y, s))
    except ValueError:
        return 0.5

def fit_heads(Xtr, ytr, Xval, yval, Xte, seed, heads=HEADS, keep_models=False):
    sc = StandardScaler().fit(Xtr)
    A, B, C = sc.transform(Xtr), sc.transform(Xval), sc.transform(Xte)
    out = {}
    for h in heads:
        t0 = time.time()
        if h == "SVM":
            cands = [(f"C={c}", SVC(kernel="rbf", C=c, class_weight="balanced")) for c in SVM_C_GRID]
            score = lambda m, X: m.decision_function(X)
        elif h == "RF":
            cands = [("300 trees", RandomForestClassifier(n_estimators=RF_TREES, class_weight="balanced", n_jobs=-1, random_state=seed))]
            score = lambda m, X: m.predict_proba(X)[:, 1]
        elif h == "LR":
            cands = [("L2", LogisticRegression(max_iter=2000, class_weight="balanced"))]
            score = lambda m, X: m.predict_proba(X)[:, 1]
        elif h == "DT":
            cands = [(f"depth={d},leaf={l}", DecisionTreeClassifier(max_depth=d, min_samples_leaf=l, class_weight="balanced", random_state=seed))
                     for d in DT_DEPTHS for l in DT_LEAVES]
            score = lambda m, X: m.predict_proba(X)[:, 1]
        else:
            raise ValueError(h)
        best = None
        for hp, m in cands:
            m.fit(A, ytr)
            v = score(m, B)
            a = _auc(yval, v)
            if best is None or a > best[0]:
                best = (a, hp, m, v)
        fit_s = time.time() - t0
        t1 = time.time()
        te = score(best[2], C)
        infer_s = (time.time() - t1) / len(C)
        out[h] = dict(val=np.asarray(best[3], np.float32), test=np.asarray(te, np.float32), hp=best[1], fit_s=fit_s, infer_s=infer_s)
        if keep_models:
            out[h]["model"] = best[2]
    if keep_models:
        out["_scaler"] = sc
    return out

def ensemble_scores(hout):
    """Soft vote of SVM + RF + LR. SVM margin is min-max normalised with VALIDATION min/max and now CLIPPED to [0,1]
    at test time (DA-2 did not clip; Sec. VII-F)."""
    need = [h for h in ("SVM", "RF", "LR") if h in hout]
    if len(need) < 2:
        return None
    parts_v, parts_t = [], []
    for h in need:
        v, t = hout[h]["val"].astype(np.float64), hout[h]["test"].astype(np.float64)
        if h == "SVM":
            lo, hi = v.min(), v.max()
            v = np.clip((v - lo) / (hi - lo + 1e-12), 0, 1); t = np.clip((t - lo) / (hi - lo + 1e-12), 0, 1)
        parts_v.append(v); parts_t.append(t)
    return np.mean(parts_v, 0).astype(np.float32), np.mean(parts_t, 0).astype(np.float32)

def make_row(b, seed, backbone, loss, head, tap, yval, yte, sval, ste, extra=None):
    sval = np.asarray(sval, np.float32); ste = np.asarray(ste, np.float32)
    thr, val_ba = find_best_threshold(yval, sval)                     # validation only
    pred = (ste.astype(np.float64) >= thr).astype(int)                # test scored once with the frozen threshold
    m = compute_metrics(yte, pred); m.pop("confusion_matrix")
    m.update(auc_metrics(yte, ste))
    row = dict(benchmark=int(b), seed=int(seed), backbone=backbone, loss=loss, head=head, tap=tap,
               method=mid(backbone, loss, head, tap), threshold=float(thr), val_ba=float(val_ba),
               val_auc=auc_metrics(yval, sval)["roc_auc"], n_val_pos=int(np.sum(yval)), n_test_pos=int(np.sum(yte)))
    row.update(m); row.update(extra or {})
    return row

def process_heads(b, seed, backbone, loss, tap, hout, yva, yte, rows, scores, ext_s=0.0, train_s=0.0, n_params=0):
    """Turn fitted heads on one tap into result rows + stored raw scores (incl. the ensemble on the late tap)."""
    for h, r in hout.items():
        if h.startswith("_"):
            continue
        extra = dict(fit_s=r["fit_s"], head_infer_s=r["infer_s"], feat_extract_s=ext_s, infer_total_s=ext_s + r["infer_s"],
                     backbone_train_s=train_s, pipeline_train_s=train_s + r["fit_s"], n_params=n_params, hp=str(r["hp"]))
        rows.append(make_row(b, seed, backbone, loss, h, tap, yva, yte, r["val"], r["test"], extra))
        scores[mid(backbone, loss, h, tap)] = (r["val"], r["test"])
    if tap == "late":
        ens = ensemble_scores(hout)
        if ens is not None:
            parts = [hout[h] for h in ("SVM", "RF", "LR") if h in hout]
            extra = dict(fit_s=sum(p["fit_s"] for p in parts), head_infer_s=sum(p["infer_s"] for p in parts), feat_extract_s=ext_s,
                         infer_total_s=ext_s + sum(p["infer_s"] for p in parts), backbone_train_s=train_s,
                         pipeline_train_s=train_s + sum(p["fit_s"] for p in parts), n_params=n_params, hp="soft-vote")
            rows.append(make_row(b, seed, backbone, loss, "ENS", tap, yva, yte, ens[0], ens[1], extra))
            scores[mid(backbone, loss, "ENS", tap)] = ens

def run_tag(b, seed, backbone, loss):
    return f"b{b}_s{seed}_{backbone}_{loss}"
def rows_path(tag):
    return os.path.join(RAW_DIR, tag + ".json")
def npz_path(tag):
    return os.path.join(RAW_DIR, tag + ".npz")
def run_done(tag):
    return os.path.exists(rows_path(tag)) and os.path.exists(npz_path(tag))

def save_run(tag, rows, scores, yva, yte):
    """npz first, JSON last: the JSON's existence marks the run as complete (safe to resume after a disconnect)."""
    arrs = dict(y_val=np.asarray(yva, np.int8), y_test=np.asarray(yte, np.int8))
    for m, (v, t) in scores.items():
        arrs[skey(m) + "__val"] = np.asarray(v, np.float32); arrs[skey(m) + "__test"] = np.asarray(t, np.float32)
    np.savez(npz_path(tag), **arrs)
    with open(rows_path(tag), "w") as f:
        json.dump(rows, f)

In [ ]:
# 11. One experiment unit: run_one(benchmark, seed, backbone, loss)   [MODIFIED: replaces the monolithic main loop]
# Serves: factorial ablation (backbone x loss x head x tap), seeds, validation-only selection, compute cost.
# Per run: seeded 85/15 split -> train CNN (shared recipe) -> end-to-end scores -> extract early/late/combined features ->
# fit SVM/RF/LR/DT (+ ensemble) -> thresholds from VALIDATION -> test scored once -> raw scores + rows saved to RAW_DIR.
def get_flops(model, img_size=IMG_SIZE):
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2_as_graph
        fn = tf.function(lambda x: model(x))
        concrete = fn.get_concrete_function(tf.TensorSpec([1, img_size, img_size, 1], tf.float32))
        _, graph_def = convert_variables_to_constants_v2_as_graph(concrete)
        with tf.Graph().as_default() as g:
            tf.graph_util.import_graph_def(graph_def, name="")
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            opts["output"] = "none"                      # silence the profiler report
            prof = tf.compat.v1.profiler.profile(graph=g, run_meta=tf.compat.v1.RunMetadata(), cmd="op", options=opts)
            return float(prof.total_float_ops)
    except Exception as e:
        print("FLOPs unavailable:", repr(e)[:120])
        return float("nan")

def _time_ms(fn, n=200, warm=10):
    for i in range(warm):
        fn(i)
    t0 = time.perf_counter()
    for i in range(n):
        fn(i)
    return (time.perf_counter() - t0) / n * 1e3

def measure_latency(b, seed, backbone, loss, model, ext, heads_by_tap, Xte, n=200):
    """Per-sample latency on PRELOADED uint8 arrays (no disk / PNG decoding), batch size 1, warm-up excluded.
    Two-stage pipelines are timed END-TO-END: extractor forward + scaler + head. RF is timed with n_jobs=1."""
    rng = np.random.default_rng(0)
    S = Xte[rng.choice(len(Xte), size=min(n + 10, len(Xte)), replace=False)]
    x1 = lambda i: tf.convert_to_tensor(S[i % len(S):i % len(S) + 1].astype(np.float32) / 255.0)
    big = tf.convert_to_tensor(Xte[:256].astype(np.float32) / 255.0)
    out = []
    base = dict(benchmark=b, seed=seed, backbone=backbone, loss=loss)
    # COMPILED graphs (tf.function, dynamic batch). Calling a Keras model eagerly at batch 1 measures per-layer Python dispatch
    # (15-60 ms here), not the network; eager time is kept as `eager_ms` for transparency only.
    comp = lambda m: tf.function(lambda x: m(x, training=False), input_signature=[tf.TensorSpec([None, IMG_SIZE, IMG_SIZE, 1], tf.float32)])
    fm = comp(model); fe = {t: comp(ext[t]) for t in ("early", "late")}
    eager_ms = _time_ms(lambda i: model(x1(i), training=False).numpy(), 50)
    e2e_b1 = _time_ms(lambda i: fm(x1(i)).numpy(), n)
    e2e_b256 = _time_ms(lambda i: fm(big).numpy(), 10, 2) / len(big)
    out.append(dict(base, head="e2e", tap="-", ext_ms=e2e_b1, head_ms=0.0, total_ms=e2e_b1, total_ms_b256=e2e_b256, eager_ms=eager_ms,
                    n_params=int(model.count_params()), flops=get_flops(model)))
    ext_ms, ext_b256 = {}, {}
    for t in ("early", "late"):
        ext_ms[t] = _time_ms(lambda i: fe[t](x1(i)).numpy(), n)
        ext_b256[t] = _time_ms(lambda i: fe[t](big).numpy(), 10, 2) / len(big)
    for tap, hout in heads_by_tap.items():
        sc = hout["_scaler"]
        taps = ["early", "late"] if tap == "combined" else [tap]
        def feats(i, taps=taps):
            return np.concatenate([fe[t](x1(i)).numpy() for t in taps], axis=1)
        fixed = sc.transform(feats(0))
        e_ms = sum(ext_ms[t] for t in taps)
        e_b256 = sum(ext_b256[t] for t in taps)
        head_ms = {}
        for h in [k for k in hout if not k.startswith("_")]:
            m = hout[h]["model"]
            old = m.get_params().get("n_jobs", None) if hasattr(m, "get_params") else None
            if old is not None:
                m.set_params(n_jobs=1)
            pred = (lambda z: m.decision_function(z)) if h == "SVM" else (lambda z: m.predict_proba(z))
            head_ms[h] = _time_ms(lambda i: pred(fixed), n)
            total = _time_ms(lambda i: pred(sc.transform(feats(i))), n)
            if old is not None:
                m.set_params(n_jobs=old)
            out.append(dict(base, head=h, tap=tap, ext_ms=e_ms, head_ms=head_ms[h], total_ms=total,
                            total_ms_b256=e_b256 + hout[h]["infer_s"] * 1e3,
                            n_params=int(model.count_params()), flops=float("nan")))
        if tap == "late" and all(k in head_ms for k in ("SVM", "RF", "LR")):
            hm = sum(head_ms[k] for k in ("SVM", "RF", "LR"))
            out.append(dict(base, head="ENS", tap=tap, ext_ms=e_ms, head_ms=hm, total_ms=e_ms + hm,
                            total_ms_b256=e_b256 + sum(hout[k]["infer_s"] for k in ("SVM", "RF", "LR")) * 1e3,
                            n_params=int(model.count_params()), flops=float("nan")))
    for t in ("early", "late"):
        out.append(dict(base, head="extractor", tap=t, ext_ms=ext_ms[t], head_ms=0.0, total_ms=ext_ms[t],
                        total_ms_b256=ext_b256[t], n_params=int(ext[t].count_params()), flops=get_flops(ext[t])))
    return out

def run_one(b, seed, backbone, loss, force=False):
    tag = run_tag(b, seed, backbone, loss)
    if run_done(tag) and not force:
        return f"skip {tag}"
    t_run = time.time()
    set_seed(seed)
    Xtr, ytr, Xva, yva, Xte, yte = get_split(b, seed)
    bs = TRAIN_CFG["batch_size"]
    train_ds = make_ds(Xtr, ytr, bs, shuffle=True, augment=TRAIN_CFG["augment"], seed=seed)
    val_ds = make_ds(Xva, yva, bs)
    model, ext = build_backbone(backbone, loss, ytr)
    n_params = int(model.count_params())
    train_s, n_ep = train_cnn(model, train_ds, val_ds)

    rows, scores = [], {}
    sval = predict_batched(model, Xva).ravel()
    t0 = time.time(); ste = predict_batched(model, Xte).ravel(); e2e_s = (time.time() - t0) / len(Xte)
    rows.append(make_row(b, seed, backbone, loss, "e2e", "-", yva, yte, sval, ste,
                         dict(train_s=train_s, n_epochs=n_ep, n_params=n_params, feat_extract_s=0.0, head_infer_s=e2e_s,
                              infer_total_s=e2e_s, backbone_train_s=train_s, pipeline_train_s=train_s, fit_s=0.0, hp="")))
    scores[mid(backbone, loss, "e2e", "-")] = (sval, ste)

    taps = ["early", "late", "combined"] if (backbone, loss) in FULL_TAP_COMBOS else ["late"]
    first_seed = (seed == SEEDS[0])
    feats, ext_s = {}, {}
    for t in (["early", "late"] if len(taps) > 1 else ["late"]):
        ftr = predict_batched(ext[t], Xtr); fva = predict_batched(ext[t], Xva)
        t0 = time.time(); fte = predict_batched(ext[t], Xte); ext_s[t] = (time.time() - t0) / len(Xte)
        feats[t] = (ftr, fva, fte)
    if "combined" in taps:
        feats["combined"] = tuple(np.concatenate([feats["early"][i], feats["late"][i]], axis=1) for i in range(3))
        ext_s["combined"] = ext_s["early"] + ext_s["late"]
    heads_by_tap = {}
    for tap in taps:
        ftr, fva, fte = feats[tap]
        hout = fit_heads(ftr, ytr, fva, yva, fte, seed, keep_models=first_seed)
        heads_by_tap[tap] = hout
        process_heads(b, seed, backbone, loss, tap, hout, yva, yte, rows, scores, ext_s=ext_s[tap], train_s=train_s, n_params=n_params)

    if first_seed:   # latency / FLOPs are measured once per (benchmark, config), on the first seed
        lat = measure_latency(b, seed, backbone, loss, model, ext, heads_by_tap, Xte)
        with open(os.path.join(RAW_DIR, "lat_" + tag + ".json"), "w") as f:
            json.dump(lat, f)
    save_run(tag, rows, scores, yva, yte)
    e2e = rows[0]
    msg = (f"{tag}: train {train_s:.0f}s ({n_ep} ep) | e2e val BA {e2e['val_ba']:.3f} test BA {e2e['balanced_accuracy']:.3f} "
           f"recall {e2e['recall']:.3f} FA {e2e['fa']} | total {time.time() - t_run:.0f}s")
    del model, ext, heads_by_tap, feats
    tf.keras.backend.clear_session(); gc.collect()
    return msg

In [ ]:
# 12. RUN EXPERIMENTS   [NEW]   (backbone x loss grid, over seeds and benchmarks; resumable)
# Pilot: QUICK_PILOT=True -> 1 seed. Full study: QUICK_PILOT=False, ideally ONE seed per Kaggle session
# (e.g. RUN_SEEDS = [2]); finished runs are skipped, so you can simply re-run this cell after a disconnect.
CONFIGS = [(bb, ls) for bb in BACKBONES for ls in LOSSES]
if RUN_PARAM_MATCHED:
    CONFIGS += [("ref_wide", ls) for ls in LOSSES]

RUN_BENCHMARKS = ALL_BENCHMARKS          # SMOKE TEST FIRST: set RUN_BENCHMARKS = [5] (smallest benchmark) and check the output line, then use all five

def run_all(seeds=None, benchmarks=None, configs=None):
    seeds = RUN_SEEDS if seeds is None else seeds
    benchmarks = RUN_BENCHMARKS if benchmarks is None else benchmarks
    configs = CONFIGS if configs is None else configs
    total = len(seeds) * len(benchmarks) * len(configs); k = 0; t0 = time.time()
    for seed in seeds:
        for b in benchmarks:
            for bb, ls in configs:
                k += 1
                print(f"[{k}/{total}] ", end="")
                print(run_one(b, seed, bb, ls), flush=True)
            release_arrays()
    print(f"Finished in {(time.time() - t0) / 60:.1f} min")

run_all()

In [ ]:
# 11b. Re-measure inference latency with compiled graphs   [NEW]  (run once, ~5-10 min; then re-run cell 20)
# The first full run timed eager Keras calls at batch 1 (17-58 ms per clip, dominated by Python layer dispatch; DA-2 reported ~1 ms).
# This retrains each backbone/loss config ONCE on one benchmark (same recipe, seed 0), refits the heads, and times the COMPILED
# extractor + scaler + head end to end. It deletes the old lat_*.json so cell 20 uses one consistent measurement.
# Latency does not depend on the trained weights; head latency depends on fitted-model size, which is why the heads are refit.
# Quote in the paper: GPU model, batch size 1, benchmark used, RF with n_jobs=1.
LAT_BENCHMARK = 2
def remeasure_latency(b=LAT_BENCHMARK, seed=0):
    for f in glob.glob(os.path.join(RAW_DIR, "lat_b*.json")):
        os.remove(f)
    set_seed(seed)
    Xtr, ytr, Xva, yva, Xte, yte = get_split(b, seed)
    train_ds = make_ds(Xtr, ytr, TRAIN_CFG["batch_size"], shuffle=True, augment=TRAIN_CFG["augment"], seed=seed)
    val_ds = make_ds(Xva, yva, TRAIN_CFG["batch_size"])
    for backbone, loss in CONFIGS:
        model, ext = build_backbone(backbone, loss, ytr)
        train_cnn(model, train_ds, val_ds)
        taps = ["early", "late", "combined"] if (backbone, loss) in FULL_TAP_COMBOS else ["late"]
        feats = {t: tuple(predict_batched(ext[t], X) for X in (Xtr, Xva, Xte)) for t in ("early", "late")}
        feats["combined"] = tuple(np.concatenate([feats["early"][i], feats["late"][i]], axis=1) for i in range(3))
        heads_by_tap = {tap: fit_heads(feats[tap][0], ytr, feats[tap][1], yva, feats[tap][2], seed, keep_models=True) for tap in taps}
        lat = measure_latency(b, seed, backbone, loss, model, ext, heads_by_tap, Xte)
        with open(os.path.join(RAW_DIR, "lat_" + run_tag(b, seed, backbone, loss) + ".json"), "w") as f:
            json.dump(lat, f)
        print(f"{backbone}/{loss}: e2e {lat[0]['total_ms']:.2f} ms (eager {lat[0]['eager_ms']:.1f} ms), batch-256 {lat[0]['total_ms_b256']:.3f} ms/clip")
        del model, ext, heads_by_tap, feats
        tf.keras.backend.clear_session(); gc.collect()
    release_arrays()

remeasure_latency()

In [ ]:
# 11c. Epoch-cap check   [NEW; optional, ~15-20 min on a T4]  (answers: "were the 40% of runs that hit the 20-epoch cap still improving?")
# Retrains end-to-end CNNs on ONE benchmark with epochs=20 vs epochs=40 (same seed => same split and same initialisation; everything else
# identical, early stopping still active) and compares validation AUC, test AUC and test BA (threshold re-selected on validation, as in
# the main pipeline). Nothing is saved into RAW_DIR, so none of the 100 main runs or any table is affected.
# Read-out: if (cap 40 - cap 20) is small compared with the seed std, the cap is not material for the paper's conclusions.
CAP_BENCHMARK = 4        # ICCAD-4 had the most cap hits (proposed/focal: 5 of 5 seeds)
CAP_SEEDS = (0, 1, 2)
CAP_CONFIGS = (("ref", "focal"), ("proposed", "focal"))
CAP_VALUES = (20, 40)

def epoch_cap_test(b=CAP_BENCHMARK, seeds=CAP_SEEDS, configs=CAP_CONFIGS, caps=CAP_VALUES):
    out, cap0 = [], TRAIN_CFG["epochs"]
    for seed in seeds:
        Xtr, ytr, Xva, yva, Xte, yte = get_split(b, seed)
        for bb, ls in configs:
            for cap in caps:
                set_seed(seed)                                        # identical split and initialisation for both caps
                train_ds = make_ds(Xtr, ytr, TRAIN_CFG["batch_size"], shuffle=True, augment=TRAIN_CFG["augment"], seed=seed)
                val_ds = make_ds(Xva, yva, TRAIN_CFG["batch_size"])
                TRAIN_CFG["epochs"] = cap
                try:
                    model, ext = build_backbone(bb, ls, ytr)
                    train_s, n_ep = train_cnn(model, train_ds, val_ds)
                finally:
                    TRAIN_CFG["epochs"] = cap0                        # never leave the shared recipe modified
                sval = predict_batched(model, Xva).ravel(); ste = predict_batched(model, Xte).ravel()
                r = make_row(b, seed, bb, ls, "e2e", "-", yva, yte, sval, ste)
                out.append(dict(benchmark=b, seed=seed, config=f"{bb}/{ls}", cap=cap, epochs_run=n_ep, hit_cap=int(n_ep >= cap),
                                train_s=round(train_s), val_auc=r["val_auc"], test_auc=r["roc_auc"], test_ba=r["balanced_accuracy"],
                                recall=r["recall"], fa=r["fa"]))
                print(f"ICCAD-{b} seed {seed} {bb}/{ls} cap {cap}: {n_ep} epochs, val AUC {r['val_auc']:.4f}, test AUC {r['roc_auc']:.4f}, "
                      f"test BA {r['balanced_accuracy']:.3f} ({train_s:.0f}s)", flush=True)
                del model, ext
                tf.keras.backend.clear_session(); gc.collect()
    release_arrays()
    return pd.DataFrame(out)

CAP = epoch_cap_test()
CAP.to_csv(os.path.join(TAB_DIR, "epoch_cap_check.csv"), index=False)
summ = CAP.groupby(["config", "cap"]).agg(epochs_run=("epochs_run", "mean"), share_hit_cap=("hit_cap", "mean"), val_auc=("val_auc", "mean"),
                                          test_auc=("test_auc", "mean"), test_ba=("test_ba", "mean"), test_ba_sd=("test_ba", "std"), train_s=("train_s", "mean"))
print("\n", summ.round(4).to_string())
for cfg, d in CAP.groupby("config"):
    lo, hi = min(CAP_VALUES), max(CAP_VALUES)
    p = d.pivot(index="seed", columns="cap", values=["test_ba", "test_auc", "val_auc"])
    print(f"\n{cfg}: effect of raising the cap {lo} -> {hi} (mean ± std over {len(p)} seeds)")
    for m in ("test_ba", "test_auc", "val_auc"):
        diff = (p[(m, hi)] - p[(m, lo)])
        print(f"   delta {m:9s} {diff.mean():+.4f} ± {diff.std(ddof=1):.4f}   (seeds improved: {(diff > 0).sum()}/{len(diff)})")
print("\nInterpretation: compare each delta with the seed-to-seed std of that metric in the main tables (BA std on ICCAD-4 is ~0.01-0.02). "
      "Deltas well inside that band => the 20-epoch cap does not change the conclusions; a clear, consistent gain for the REFERENCE CNN "
      "(esp. on BA/AUC) would mean its numbers are slightly conservative and should be mentioned in the limitations.")

In [ ]:
# 13. OPTIONAL controls: random-init backbone + ImageNet VGG16 features   [NEW]  (off by default)
# Serves: showing that TRAINING the backbone matters (random-init control) and a measured transfer-learning comparison
# (replaces the unmeasured "128x faster than VGG16" claim, which borrowed a 182 ms figure from [1] measured on other hardware).
RUN_CONTROLS = False

def run_feature_control(b, seed, name, feat_fn, n_params=0):
    tag = run_tag(b, seed, name, "none")
    if run_done(tag):
        return f"skip {tag}"
    set_seed(seed)
    Xtr, ytr, Xva, yva, Xte, yte = get_split(b, seed)
    ftr, fva = feat_fn(Xtr), feat_fn(Xva)
    t0 = time.time(); fte = feat_fn(Xte); ext_s = (time.time() - t0) / len(Xte)
    hout = fit_heads(ftr, ytr, fva, yva, fte, seed)
    rows, scores = [], {}
    process_heads(b, seed, name, "none", "late", hout, yva, yte, rows, scores, ext_s=ext_s, train_s=0.0, n_params=n_params)
    save_run(tag, rows, scores, yva, yte)
    return f"{tag}: done ({len(rows)} methods)"

def randinit_fn(seed):
    set_seed(seed)
    model, ext = build_backbone("proposed", "focal", np.array([0, 1]))
    return (lambda X: predict_batched(ext["late"], X)), int(model.count_params())

def vgg16_fn():
    base = tf.keras.applications.VGG16(weights="imagenet", include_top=False, pooling="avg", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    def fn(X):
        outs = []
        for i in range(0, len(X), 256):
            xb = np.repeat(X[i:i + 256].astype(np.float32), 3, axis=-1)
            outs.append(base(tf.keras.applications.vgg16.preprocess_input(xb), training=False).numpy())
        return np.concatenate(outs, axis=0)
    return fn, int(base.count_params())

if RUN_CONTROLS:
    for seed in RUN_SEEDS:
        for b in BENCHMARKS:
            fn, npar = randinit_fn(seed); print(run_feature_control(b, seed, "rand_init", fn, npar))
            fn, npar = vgg16_fn();        print(run_feature_control(b, seed, "vgg16", fn, npar))
            release_arrays(); tf.keras.backend.clear_session()

## Analysis cells (14 onward)
Everything below reads only `outputs/raw` (saved scores + rows). It needs **no GPU and no retraining**, so you can re-run it freely after each seed finishes.

In [ ]:
# 14. Load all runs; results tables as mean ± std over seeds   [MODIFIED: replaces the single-seed tables]
# Serves: "report mean ± std over 3-5 seeds". Everything below reads only RAW_DIR (no retraining).
def load_results():
    rows = []
    for f in sorted(glob.glob(os.path.join(RAW_DIR, "b*_s*_*.json"))):
        with open(f) as fh:
            rows += json.load(fh)
    return pd.DataFrame(rows)

df = load_results()
assert len(df), f"No finished runs found in {RAW_DIR}. Run cell 12 first."
SEEDS_PRESENT = sorted(df.seed.unique().tolist())
BENCHMARKS = sorted(df.benchmark.unique().tolist())      # analysis uses the benchmarks that have finished runs (run_all uses ALL_BENCHMARKS)
if BENCHMARKS != ALL_BENCHMARKS:
    print(f"NOTE: only benchmarks {BENCHMARKS} have results; 'Average' columns and the across-benchmark tests cover these only.")
print(f"{len(df)} result rows | seeds present: {SEEDS_PRESENT} | benchmarks: {sorted(df.benchmark.unique().tolist())} | methods: {df.method.nunique()}")
if len(SEEDS_PRESENT) < 3:
    print("NOTE: fewer than 3 seeds -> std / CIs across seeds / paired tests are not meaningful yet (pilot mode).")

ROW = df.set_index(["benchmark", "seed", "method"]).sort_index()

_NPZ = {}
def load_scores(b, seed, backbone, loss):
    key = (b, seed, backbone, loss)
    if key not in _NPZ:
        if len(_NPZ) >= 8:
            _NPZ.pop(next(iter(_NPZ)))
        z = np.load(npz_path(run_tag(b, seed, backbone, loss)))
        _NPZ[key] = {k: z[k] for k in z.files}
    return _NPZ[key]

def get_scores(method, b, seed):
    """(val_scores, val_labels, test_scores, test_labels) for one stored method."""
    bb, ls, h, tap = parse_mid(method)
    z = load_scores(b, seed, bb, ls); k = skey(method)
    return z[k + "__val"].astype(np.float64), z["y_val"].astype(int), z[k + "__test"].astype(np.float64), z["y_test"].astype(int)

def main_methods(d=None):
    d = df if d is None else d
    want = [mid("ref", "focal", "e2e", "-"), mid("proposed", "focal", "e2e", "-")]
    for tap in ("early", "late", "combined"):
        for h in HEADS:
            want.append(mid(*MAIN_CONFIG, h, tap))
    want.append(mid(*MAIN_CONFIG, "ENS", "late"))
    present = set(d.method)
    return [m for m in want if m in present]

def seed_summary(d, metric, methods=None):
    d = d if methods is None else d[d.method.isin(methods)]
    g = d.groupby(["method", "benchmark"])[metric]
    mean, std = g.mean().unstack("benchmark"), g.std(ddof=1).unstack("benchmark")
    per_seed = d.groupby(["method", "seed"])[metric].mean().unstack("seed")     # benchmark-average within each seed first
    mean["Average"] = per_seed.mean(axis=1); std["Average"] = per_seed.std(axis=1, ddof=1)
    if methods is not None:
        mean, std = mean.reindex(methods), std.reindex(methods)
    return mean, std

def fmt_table(mean, std, digits=3):
    out = pd.DataFrame(index=[pretty(m) for m in mean.index], columns=[f"BM-{c}" if isinstance(c, (int, np.integer)) else c for c in mean.columns], dtype=object)
    for i, m in enumerate(mean.index):
        for j, c in enumerate(mean.columns):
            mu, sd = mean.iloc[i, j], std.iloc[i, j]
            out.iloc[i, j] = "-" if pd.isna(mu) else (f"{mu:.{digits}f}" if pd.isna(sd) else f"{mu:.{digits}f} ± {sd:.{digits}f}")
    return out

METRICS = [("balanced_accuracy", "Balanced accuracy", 3), ("recall", "Recall (hotspot detection rate)", 3), ("fa", "False alarms (# NHS clips flagged)", 0),
           ("precision", "Precision", 3), ("specificity", "Specificity", 3), ("f1", "F1", 3), ("roc_auc", "ROC-AUC", 3), ("average_precision", "PR-AUC / AP", 3)]
MAIN = main_methods()
for metric, title, dg in METRICS:
    mean, std = seed_summary(df, metric, MAIN)
    tab = fmt_table(mean, std, dg)
    tab.to_csv(os.path.join(TAB_DIR, f"main_{metric}.csv"))
    fmt_table(*seed_summary(df, metric), dg).to_csv(os.path.join(TAB_DIR, f"all_methods_{metric}.csv"))
    print(f"\n=== {title}: mean ± std over seeds {SEEDS_PRESENT} ===")
    print(tab.to_string())

In [ ]:
# 15. Separating the effects of BACKBONE, LOSS and HEAD (+ figures)   [NEW]
# Serves: "separate the effects of backbone, classical head and focal loss". Each effect changes ONE factor and holds
# the others fixed; the unit is balanced accuracy averaged over benchmarks, paired by seed.
BAS = df.groupby(["method", "seed"])["balanced_accuracy"].mean().unstack("seed")

def paired_effect(A, B, factor, context):
    if A not in BAS.index or B not in BAS.index:
        return None
    a, b = BAS.loc[A].dropna(), BAS.loc[B].dropna()
    common = a.index.intersection(b.index)
    d = (b[common] - a[common]).values
    p = stats.ttest_rel(b[common], a[common]).pvalue if len(d) >= 3 else float("nan")
    return dict(factor=factor, context=context, A=pretty(A), B=pretty(B), delta_BA=float(d.mean()),
                std_over_seeds=float(d.std(ddof=1)) if len(d) > 1 else float("nan"), n_seeds=len(d), p_paired_t=float(p))

TAPS_HEADS = [("e2e", "-")] + [(h, "late") for h in HEADS + ["ENS"]]
_eff = []
for loss in LOSSES:                                      # BACKBONE effect (reference -> proposed), loss and head fixed
    for h, t in TAPS_HEADS:
        _eff.append(paired_effect(mid("ref", loss, h, t), mid("proposed", loss, h, t), "backbone: ref -> proposed", f"loss={loss}, head={h}"))
for bb in BACKBONES:                                     # LOSS effect (weighted BCE -> focal), backbone and head fixed
    for h, t in TAPS_HEADS:
        _eff.append(paired_effect(mid(bb, "bce", h, t), mid(bb, "focal", h, t), "loss: BCE -> focal", f"backbone={bb}, head={h}"))
for bb in BACKBONES:                                     # HEAD effect (end-to-end -> classical on late features), backbone and loss fixed
    for loss in LOSSES:
        for h in HEADS + ["ENS"]:
            _eff.append(paired_effect(mid(bb, loss, "e2e", "-"), mid(bb, loss, h, "late"), "head: end-to-end -> classical", f"backbone={bb}, loss={loss}, head={h}"))
if "ref_wide" in set(df.backbone):                       # capacity control: is it the architecture or just more parameters?
    for loss in LOSSES:
        for h, t in TAPS_HEADS:
            _eff.append(paired_effect(mid("ref_wide", loss, h, t), mid("proposed", loss, h, t), "architecture at matched params: ref-wide -> proposed", f"loss={loss}, head={h}"))
if "rand_init" in set(df.backbone):
    for h in HEADS:
        _eff.append(paired_effect(mid("rand_init", "none", h, "late"), mid(*MAIN_CONFIG, h, "late"), "training: random-init -> trained backbone", f"head={h}"))
EFFECTS = pd.DataFrame([e for e in _eff if e is not None])
EFFECTS.to_csv(os.path.join(TAB_DIR, "effects_factorial.csv"), index=False)
print("Effect of changing ONE factor on benchmark-averaged balanced accuracy (positive = second option better):")
print(EFFECTS.round(4).to_string(index=False))

# Factorial summary (balanced accuracy, mean over seeds) for the report: rows = backbone x loss x head
mean_all, std_all = seed_summary(df, "balanced_accuracy", [mid(bb, ls, h, t) for bb in BACKBONES for ls in LOSSES for h, t in TAPS_HEADS
                                                          if mid(bb, ls, h, t) in set(df.method)])
fmt_table(mean_all, std_all).to_csv(os.path.join(TAB_DIR, "factorial_balanced_accuracy.csv"))

# ---- Fig. 4: balanced accuracy per benchmark with seed-std error bars ----
mean, std = seed_summary(df, "balanced_accuracy", MAIN)
fig, ax = plt.subplots(figsize=(11, 4.2)); w = 0.8 / len(MAIN)
for i, m in enumerate(MAIN):
    ax.bar(np.arange(len(BENCHMARKS)) + i * w, [mean.loc[m, b] for b in BENCHMARKS], w,
           yerr=[0 if pd.isna(std.loc[m, b]) else std.loc[m, b] for b in BENCHMARKS], capsize=1.5, label=pretty(m))
ax.set_xticks(np.arange(len(BENCHMARKS)) + 0.4 - w / 2); ax.set_xticklabels([f"ICCAD-{b}" for b in BENCHMARKS])
ax.set_ylabel("Balanced accuracy"); ax.set_ylim(0.4, 1.0); ax.legend(fontsize=6, ncol=2, loc="lower right")
ax.set_title(f"Balanced accuracy, mean ± std over seeds {SEEDS_PRESENT}"); fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig4_balanced_accuracy.png"), dpi=200); plt.show()

# ---- Fig. 6: row-normalised confusion matrices, counts pooled over seeds ----
REP = [m for m in [mid("ref", "focal", "e2e", "-"), mid("proposed", "focal", "e2e", "-")] + [mid(*MAIN_CONFIG, h, "late") for h in ("SVM", "RF", "LR", "DT")] if m in set(df.method)]
fig, axes = plt.subplots(len(REP), len(BENCHMARKS), figsize=(2.3 * len(BENCHMARKS), 2.2 * len(REP)), squeeze=False)
for i, m in enumerate(REP):
    for j, b in enumerate(BENCHMARKS):
        s = df[(df.method == m) & (df.benchmark == b)][["tp", "fn", "fp", "tn"]].sum()
        cm = np.array([[s.tp, s.fn], [s.fp, s.tn]], float); pc = cm / np.maximum(cm.sum(1, keepdims=True), 1) * 100
        ax = axes[i, j]; ax.imshow(pc, cmap="Blues", vmin=0, vmax=100)
        for r in range(2):
            for c in range(2):
                ax.text(c, r, f"{pc[r, c]:.1f}%", ha="center", va="center", fontsize=6, color="white" if pc[r, c] > 50 else "black")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["HS", "NHS"], fontsize=6); ax.set_yticks([0, 1]); ax.set_yticklabels(["HS", "NHS"], fontsize=6)
        if i == 0: ax.set_title(f"ICCAD-{b}", fontsize=8)
        if j == 0: ax.set_ylabel(pretty(m), fontsize=5)
fig.suptitle("Row-normalised confusion matrices (counts pooled over seeds)", fontsize=9); fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig6_confusion_matrices.png"), dpi=200); plt.show()

In [ ]:
# 16. Confidence intervals   [NEW]
# Serves: "add confidence intervals".
#  * Test-set sampling uncertainty with the decision threshold FROZEN: stratified bootstrap (hotspots and non-hotspots resampled
#    separately). For a fixed classifier this is exactly TP* ~ Binomial(n_pos, TPR), FP* ~ Binomial(n_neg, FPR), so we draw from
#    that directly. Draws from all seeds are POOLED, so the CI also contains training/threshold-selection (seed) variance.
#  * Recall / specificity additionally get exact Clopper-Pearson intervals (first seed).
#  * ROC-AUC gets a DeLong interval (first seed).
def clopper_pearson(k, n, alpha=0.05):
    lo = beta.ppf(alpha / 2, k, n - k + 1) if k > 0 else 0.0
    hi = beta.ppf(1 - alpha / 2, k + 1, n - k) if k < n else 1.0
    return float(lo), float(hi)

def bootstrap_from_counts(tp, fn, fp, tn, B, rng):
    n_pos, n_neg = tp + fn, fp + tn
    tp_b = rng.binomial(n_pos, tp / n_pos, B); fp_b = rng.binomial(n_neg, fp / n_neg, B)
    rec = tp_b / n_pos; spec = 1 - fp_b / n_neg
    prec = tp_b / np.maximum(tp_b + fp_b, 1)
    f1 = 2 * tp_b / np.maximum(2 * tp_b + fp_b + (n_pos - tp_b), 1)
    return dict(balanced_accuracy=(rec + spec) / 2, recall=rec, specificity=spec, precision=prec, f1=f1, fa=fp_b.astype(float))

def delong_auc_ci(y, s, alpha=0.05):
    y = np.asarray(y).astype(int); s = np.asarray(s, float)
    pos, neg = s[y == 1], s[y == 0]; m, n = len(pos), len(neg)
    tx, ty, tz = rankdata(pos), rankdata(neg), rankdata(np.concatenate([pos, neg]))
    auc = (tz[:m].sum() / m - (m + 1) / 2) / n
    v10 = (tz[:m] - tx) / n; v01 = 1 - (tz[m:] - ty) / m
    se = math.sqrt(np.var(v10, ddof=1) / m + np.var(v01, ddof=1) / n)
    z = stats.norm.ppf(1 - alpha / 2)
    return float(auc), float(max(0, auc - z * se)), float(min(1, auc + z * se))

CI_B = 4000
def ci_table(methods, B=CI_B, seed0=0):
    rng = np.random.default_rng(seed0); out = []
    for m in methods:
        for b in BENCHMARKS:
            d = df[(df.method == m) & (df.benchmark == b)]
            if d.empty:
                continue
            per_seed = max(B // len(d), 200)
            draws = {k: [] for k in ("balanced_accuracy", "recall", "specificity", "precision", "f1", "fa")}
            for _, r in d.iterrows():
                bs = bootstrap_from_counts(int(r.tp), int(r.fn), int(r.fp), int(r.tn), per_seed, rng)
                for k in draws: draws[k].append(bs[k])
            first = d.sort_values("seed").iloc[0]
            row = dict(method=pretty(m), benchmark=b, n_seeds=len(d), n_pos_test=int(first.tp + first.fn), n_neg_test=int(first.fp + first.tn))
            for k, v in draws.items():
                v = np.concatenate(v)
                row[f"{k}_mean"] = d[k].mean(); row[f"{k}_lo"], row[f"{k}_hi"] = np.percentile(v, [2.5, 97.5])
            row["recall_CP_lo"], row["recall_CP_hi"] = clopper_pearson(int(first.tp), int(first.tp + first.fn))
            row["spec_CP_lo"], row["spec_CP_hi"] = clopper_pearson(int(first.tn), int(first.tn + first.fp))
            _, yv, st, yt = get_scores(m, b, int(first.seed))
            row["auc_first_seed"], row["auc_lo"], row["auc_hi"] = delong_auc_ci(yt, st)
            out.append(row)
    return pd.DataFrame(out)

CI = ci_table(MAIN)
CI.to_csv(os.path.join(TAB_DIR, "confidence_intervals_main.csv"), index=False)
for k in ("balanced_accuracy", "recall"):
    t = CI.assign(**{k: [f"{a:.3f} [{l:.3f}, {h:.3f}]" for a, l, h in zip(CI[f"{k}_mean"], CI[f"{k}_lo"], CI[f"{k}_hi"])]})
    print(f"\n=== {k}: mean over seeds [95% CI] ===")
    print(t.pivot(index="method", columns="benchmark", values=k).reindex([pretty(m) for m in MAIN]).to_string())
print("\nICCAD-5 recall, exact Clopper-Pearson (first seed):")
print(CI[CI.benchmark == 5][["method", "n_pos_test", "recall_mean", "recall_CP_lo", "recall_CP_hi"]].round(3).to_string(index=False))

In [ ]:
# 17. Statistical tests for "method A beats method B"   [NEW]
# Serves: "use a statistical test when claiming that one method beats another".
# Per benchmark and seed (same test images => paired):
#   * paired stratified bootstrap of delta-BA (shared resamples) -> CI of the difference
#   * exact McNemar on the HS-only and NHS-only subsets (are the two classifiers' errors different?)
# Across seeds, on the benchmark-averaged BA: paired t-test (n>=3) and Wilcoxon signed-rank (only when n_seeds>=6, because with
# n=5 the smallest possible exact two-sided p is 0.0625 and can never reach 0.05), Holm-corrected over all comparisons.
SIG_B = 1000

def paired_delta_bootstrap(pa, pb, y, B=SIG_B, chunk=250, rng=None):
    rng = np.random.default_rng(0) if rng is None else rng
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    a_pos, b_pos = pa[pos].astype(np.int32), pb[pos].astype(np.int32)
    a_neg, b_neg = pa[neg].astype(np.int32), pb[neg].astype(np.int32)
    out, done = [], 0
    while done < B:
        c = min(chunk, B - done)
        ip = rng.integers(0, len(pos), (c, len(pos))); ineg = rng.integers(0, len(neg), (c, len(neg)))
        ba_a = (a_pos[ip].mean(1) + 1 - a_neg[ineg].mean(1)) / 2
        ba_b = (b_pos[ip].mean(1) + 1 - b_neg[ineg].mean(1)) / 2
        out.append(ba_b - ba_a); done += c
    return np.concatenate(out)

def mcnemar_log10p(pa, pb, y, subset):
    """Exact two-sided McNemar test on the discordant pairs, returned as log10(p). (A plain float p underflows to 0.0 when the two
    classifiers' false-alarm counts differ by thousands; log10(p) never does.)  subset=1: hotspots only; subset=0: non-hotspots only."""
    mask = (y == subset)
    ca, cb = (pa[mask] == subset), (pb[mask] == subset)
    n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    n, k = n01 + n10, min(n01, n10)
    if n == 0 or 2 * k >= n:
        return 0.0                                   # p = 1
    return float(max(-300.0, min(0.0, np.log10(2.0) + stats.binom.logcdf(k, n, 0.5) / np.log(10.0))))   # floored: logcdf can return -inf

def fmt_p(log10p):
    return "<1e-10" if log10p < -10 else f"{10 ** log10p:.3g}"

def holm_adjust(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def test_preds(b, seed, method):
    _, _, st, yt = get_scores(method, b, seed)
    return (st >= ROW.loc[(b, seed, method), "threshold"]).astype(np.int8), yt

def build_comparisons():
    E2E = mid("ref", "focal", "e2e", "-")      # the common reference baseline
    comps = [("backbone (end-to-end, focal)", mid("ref", "focal", "e2e", "-"), mid("proposed", "focal", "e2e", "-")),
             ("loss (proposed, end-to-end): BCE -> focal", mid("proposed", "bce", "e2e", "-"), mid("proposed", "focal", "e2e", "-")),
             ("loss (reference, end-to-end): BCE -> focal", mid("ref", "bce", "e2e", "-"), mid("ref", "focal", "e2e", "-"))]
    for h in HEADS + ["ENS"]:
        comps.append((f"head ({h}, late) vs proposed end-to-end", mid(*MAIN_CONFIG, "e2e", "-"), mid(*MAIN_CONFIG, h, "late")))
        comps.append((f"head ({h}, late) vs REFERENCE baseline", E2E, mid(*MAIN_CONFIG, h, "late")))
        comps.append((f"backbone at fixed head ({h}, late)", mid("ref", "focal", h, "late"), mid("proposed", "focal", h, "late")))
    for h in HEADS:
        comps.append((f"tap ({h}): early -> late", mid(*MAIN_CONFIG, h, "early"), mid(*MAIN_CONFIG, h, "late")))
    present = set(df.method)
    return [c for c in comps if c[1] in present and c[2] in present]

COMPARISONS = build_comparisons()
rng_sig = np.random.default_rng(12345)
detail, across = [], []
for name, A, Bm in COMPARISONS:
    ba_a_seed, ba_b_seed = [], []
    for seed in SEEDS_PRESENT:
        ba_a, ba_b = [], []
        for b in BENCHMARKS:
            if (b, seed, A) in ROW.index and (b, seed, Bm) in ROW.index:
                ba_a.append(ROW.loc[(b, seed, A), "balanced_accuracy"]); ba_b.append(ROW.loc[(b, seed, Bm), "balanced_accuracy"])
        if len(ba_a) == len(BENCHMARKS):
            ba_a_seed.append(np.mean(ba_a)); ba_b_seed.append(np.mean(ba_b))
    for b in BENCHMARKS:
        deltas, point, p_hs, p_nhs = [], [], [], []
        for seed in SEEDS_PRESENT:
            if (b, seed, A) not in ROW.index or (b, seed, Bm) not in ROW.index:
                continue
            pa, y = test_preds(b, seed, A); pb, _ = test_preds(b, seed, Bm)
            deltas.append(paired_delta_bootstrap(pa, pb, y, rng=rng_sig))
            point.append(ROW.loc[(b, seed, Bm), "balanced_accuracy"] - ROW.loc[(b, seed, A), "balanced_accuracy"])
            p_hs.append(mcnemar_log10p(pa, pb, y, 1)); p_nhs.append(mcnemar_log10p(pa, pb, y, 0))
        if deltas:
            lo, hi = np.percentile(np.concatenate(deltas), [2.5, 97.5])
            detail.append(dict(comparison=name, A=pretty(A), B=pretty(Bm), benchmark=b, delta_BA_mean=np.mean(point),
                               delta_BA_std=np.std(point, ddof=1) if len(point) > 1 else float("nan"), ci_lo=lo, ci_hi=hi,
                               seeds_B_better=int(np.sum(np.array(point) > 0)), n_seeds=len(point),
                               McNemar_p_HS=fmt_p(float(np.median(p_hs))), McNemar_p_NHS=fmt_p(float(np.median(p_nhs))),
                               McNemar_log10p_HS=float(np.median(p_hs)), McNemar_log10p_NHS=float(np.median(p_nhs))))
    if len(ba_a_seed) >= 2:
        d = np.array(ba_b_seed) - np.array(ba_a_seed)
        t_p = stats.ttest_rel(ba_b_seed, ba_a_seed).pvalue if len(d) >= 3 else float("nan")
        w_p = stats.wilcoxon(ba_b_seed, ba_a_seed).pvalue if len(d) >= 6 and np.any(d != 0) else float("nan")
        across.append(dict(comparison=name, A=pretty(A), B=pretty(Bm), n_seeds=len(d), delta_BA_avg_over_benchmarks=d.mean(),
                           std_over_seeds=d.std(ddof=1), p_paired_t=t_p, p_wilcoxon=w_p))
SIG_DETAIL = pd.DataFrame(detail)
SIG_ACROSS = pd.DataFrame(across)
if len(SIG_ACROSS):
    for col in ("p_paired_t", "p_wilcoxon"):
        ok = SIG_ACROSS[col].notna()
        SIG_ACROSS.loc[ok, col + "_holm"] = holm_adjust(SIG_ACROSS.loc[ok, col].values)
SIG_DETAIL.to_csv(os.path.join(TAB_DIR, "significance_per_benchmark.csv"), index=False)
SIG_ACROSS.to_csv(os.path.join(TAB_DIR, "significance_across_seeds.csv"), index=False)
print("=== Across seeds (benchmark-averaged BA, B minus A; Holm-corrected over the comparisons listed) ===")
print(SIG_ACROSS.round(4).to_string(index=False))
print("\n=== Per benchmark: delta-BA with paired-bootstrap 95% CI and McNemar p (HS-only / NHS-only), B minus A ===")
cols = ["comparison", "benchmark", "delta_BA_mean", "ci_lo", "ci_hi", "seeds_B_better", "n_seeds", "McNemar_p_HS", "McNemar_p_NHS"]
print(SIG_DETAIL[cols].round(4).to_string(index=False))
print("\nRead-out rule used in the paper: call B 'better' than A only if the per-benchmark CI excludes 0 AND it holds in most seeds; "
      "otherwise write 'no significant difference'.")

In [ ]:
# 18. Calibration   [NEW]
# Serves: "report calibration". Metrics: ECE (10 equal-count bins), Brier score, reliability diagrams.
# Three views per method: RAW score (only for probability-like outputs), PLATT (sigmoid fitted on VALIDATION scores only), and
# PLATT + PRIOR SHIFT. The prior-shift view matters: validation keeps the TRAINING hotspot ratio, but the official TEST ratio is far
# more skewed (e.g. ICCAD-5: ~1:104 train vs ~1:471 test), so even a perfectly calibrated-on-validation model is mis-calibrated on test
# unless the odds are rescaled by the known test/validation prior ratio (Saerens et al. 2002). Class-balanced weights and focal loss
# also distort raw probabilities on purpose. With ~4-15 validation hotspots Platt fits are noisy: read ECE as indicative only.
def ece_quantile(y, p, n_bins=10):
    order = np.argsort(p); y, p = np.asarray(y)[order], np.asarray(p)[order]
    return float(sum(len(ix) / len(p) * abs(y[ix].mean() - p[ix].mean()) for ix in np.array_split(np.arange(len(p)), n_bins)))

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def calib_features(method, s):
    return s if parse_mid(method)[2] in ("SVM", "ENS") else logit(s)

def calibration_row(method, b, seed):
    sv, yv, st, yt = get_scores(method, b, seed)
    head = parse_mid(method)[2]
    row = dict(method=method, benchmark=b, seed=seed)
    if head not in ("SVM", "ENS"):                                  # raw outputs are probabilities
        row["ece_raw"], row["brier_raw"] = ece_quantile(yt, st), float(np.mean((st - yt) ** 2))
    else:
        row["ece_raw"] = row["brier_raw"] = float("nan")
    platt = LogisticRegression(C=1e6, max_iter=2000).fit(calib_features(method, sv).reshape(-1, 1), yv)
    z = platt.decision_function(calib_features(method, st).reshape(-1, 1))
    p_cal = sigmoid(z)
    row["ece_platt"], row["brier_platt"] = ece_quantile(yt, p_cal), float(np.mean((p_cal - yt) ** 2))
    pi_v, pi_t = yv.mean(), yt.mean()
    p_ps = sigmoid(z + (np.log(pi_t / (1 - pi_t)) - np.log(pi_v / (1 - pi_v))))
    row["ece_platt_prior"], row["brier_platt_prior"] = ece_quantile(yt, p_ps), float(np.mean((p_ps - yt) ** 2))
    return row, (st, p_cal, p_ps, yt)

CAL_METHODS = [m for m in [mid("ref", "focal", "e2e", "-"), mid("proposed", "focal", "e2e", "-")] +
               [mid(*MAIN_CONFIG, h, "late") for h in HEADS + ["ENS"]] if m in set(df.method)]
cal_rows = []
for m in CAL_METHODS:
    for b in BENCHMARKS:
        for seed in SEEDS_PRESENT:
            if (b, seed, m) in ROW.index:
                cal_rows.append(calibration_row(m, b, seed)[0])
CAL = pd.DataFrame(cal_rows)
CAL.to_csv(os.path.join(TAB_DIR, "calibration_rows.csv"), index=False)
for metric in ("ece_raw", "ece_platt", "ece_platt_prior", "brier_raw", "brier_platt_prior"):
    mean, std = seed_summary(CAL, metric, CAL_METHODS)
    t = fmt_table(mean, std, 4); t.to_csv(os.path.join(TAB_DIR, f"calibration_{metric}.csv"))
    print(f"\n=== {metric}: mean ± std over seeds (lower is better) ===")
    print(t.to_string())

# Reliability diagrams (first seed): raw vs Platt+prior-shift, log-log because test probabilities are tiny
fig, axes = plt.subplots(2, 2, figsize=(9, 7))
for r_i, b in enumerate([2, 5]):
    for c_i, (view, lab) in enumerate([(0, "raw score"), (2, "Platt + prior shift")]):
        ax = axes[r_i, c_i]; ax.plot([1e-5, 1], [1e-5, 1], "k--", lw=0.7)
        for m in CAL_METHODS:
            if parse_mid(m)[2] in ("SVM", "ENS") and view == 0:
                continue
            _, (st, p_cal, p_ps, yt) = calibration_row(m, b, SEEDS_PRESENT[0])
            p = (st, p_cal, p_ps)[view]; order = np.argsort(p)
            xs, ys = [], []
            for ix in np.array_split(order, 10):
                xs.append(max(p[ix].mean(), 1e-5)); ys.append(max(yt[ix].mean(), 1e-5))
            ax.plot(xs, ys, marker="o", ms=3, lw=1, label=pretty(m))
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(f"ICCAD-{b}: {lab}", fontsize=9)
        ax.set_xlabel("mean predicted probability (10 equal-count bins)"); ax.set_ylabel("observed hotspot fraction")
        if r_i == 0 and c_i == 0: ax.legend(fontsize=5)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "fig_calibration.png"), dpi=200); plt.show()

In [ ]:
# 19. Threshold sensitivity   [NEW; REVISED after the first full run]
# Serves: "report threshold sensitivity". The threshold is ALWAYS chosen on validation (cell 8); here we only study, on the test
# set, how much the result moves when that threshold is perturbed (analysis only, never used for selection).
#  * perturbation = RELATIVE change of the validation false-positive rate: the threshold that gives m x (validation FPR at the chosen
#    threshold), m in {0.25, 0.5, 0.75, 1, 1.5, 2, 4}. "Halve / double the FPR" means the same thing at an operating point of 0.5% FPR as at 15%.
#    (v1 shifted whole-distribution percentiles, then v2 absolute FPR points. Both were mis-scaled for 0.2-20% hotspot prevalence:
#    curves fell to BA = 0.5 within a few points, which measures the metric's scale, not the model.)
#  * regret: oracle test-BA (best threshold on test, unattainable) minus test-BA at the validation-selected threshold
#  * alternative operating points chosen on validation: recall >= 0.95, and FPR <= 1%
#  * caveat: a Decision Tree outputs only a handful of distinct scores, so any threshold perturbation jumps between extremes (BA -> 0.5).
MULTS = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 4.0])

def sensitivity_row(method, b, seed):
    sv, yv, st, yt = get_scores(method, b, seed)
    thr = ROW.loc[(b, seed, method), "threshold"]
    neg = np.sort(sv[yv == 0])
    fpr0 = max(float((neg >= thr).mean()), 1.0 / len(neg))           # validation FPR at the chosen threshold (floored at one negative)
    thr_m = np.quantile(neg, np.clip(1.0 - fpr0 * MULTS, 0, 1))
    tpr, tnr, ba = ba_curve(yt, st, thr_m)
    ba0 = float(ba_curve(yt, st, [thr])[2][0])
    _, _, ba_all = ba_curve(yt, st, candidate_thresholds(st, 1000))
    band = ba[(MULTS >= 0.5) & (MULTS <= 2.0)]
    row = dict(method=method, benchmark=b, seed=seed, val_fpr_at_thr=fpr0, ba_at_val_thr=ba0,
               ba_half_fpr=float(ba[MULTS == 0.5][0]), ba_double_fpr=float(ba[MULTS == 2.0][0]),
               ba_worst_half_double=float(min(band.min(), ba0)), ba_range_half_double=float(max(band.max(), ba0) - min(band.min(), ba0)),
               oracle_test_ba=float(ba_all.max()), regret=float(ba_all.max() - ba0))
    for kind, target, tag in (("recall", 0.95, "rec95"), ("fpr", 0.01, "fpr1")):
        t = operating_threshold(kind, yv, sv, target)
        tp, fn, fp, tn = confusion(yt, (st >= t).astype(int))
        row[f"{tag}_recall"] = tp / max(tp + fn, 1); row[f"{tag}_fa"] = fp; row[f"{tag}_ba"] = (tp / max(tp + fn, 1) + tn / max(tn + fp, 1)) / 2
    return row, (MULTS, tpr, tnr, ba)

SENS_METHODS = [m for m in [mid("ref", "focal", "e2e", "-"), mid("proposed", "focal", "e2e", "-")] +
                [mid(*MAIN_CONFIG, h, "late") for h in HEADS + ["ENS"]] if m in set(df.method)]
sens_rows, curves = [], {}
for m in SENS_METHODS:
    for b in BENCHMARKS:
        for seed in SEEDS_PRESENT:
            if (b, seed, m) in ROW.index:
                r, c = sensitivity_row(m, b, seed); sens_rows.append(r); curves[(m, b, seed)] = c
SENS = pd.DataFrame(sens_rows)
SENS.to_csv(os.path.join(TAB_DIR, "threshold_sensitivity_rows.csv"), index=False)
for metric, title in (("ba_at_val_thr", "BA at the validation-selected threshold"),
                      ("ba_half_fpr", "BA when the validation FPR is HALVED (threshold raised)"),
                      ("ba_double_fpr", "BA when the validation FPR is DOUBLED (threshold lowered)"),
                      ("ba_worst_half_double", "Worst BA over 0.5x .. 2x validation FPR"),
                      ("regret", "Selection regret = oracle test BA − BA at validation threshold"),
                      ("rec95_fa", "False alarms at the validation threshold giving recall >= 0.95"),
                      ("rec95_recall", "Test recall at that threshold"), ("fpr1_recall", "Test recall at the validation threshold giving FPR <= 1%")):
    mean, std = seed_summary(SENS, metric, SENS_METHODS)
    t = fmt_table(mean, std, 0 if metric.endswith("_fa") else 3); t.to_csv(os.path.join(TAB_DIR, f"sens_{metric}.csv"))
    print(f"\n=== {title} (mean ± std over seeds) ===")
    print(t.to_string())

fig, axes = plt.subplots(1, len(BENCHMARKS), figsize=(3.2 * len(BENCHMARKS), 3.2), sharey=True)
for j, b in enumerate(BENCHMARKS):
    for m in SENS_METHODS:
        cs = [curves[(m, b, s)][3] for s in SEEDS_PRESENT if (m, b, s) in curves]
        if cs: axes[j].plot(MULTS, np.mean(cs, 0), marker="o", ms=2.5, lw=1, label=pretty(m))
    axes[j].set_xscale("log", base=2); axes[j].set_xticks(MULTS); axes[j].set_xticklabels(["¼", "½", "¾", "1", "1.5", "2", "4"], fontsize=7)
    axes[j].axvline(1, color="k", lw=0.5); axes[j].set_title(f"ICCAD-{b}", fontsize=9); axes[j].set_xlabel("validation FPR multiplier")
axes[0].set_ylabel("test balanced accuracy"); axes[0].legend(fontsize=5)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "fig_threshold_sensitivity.png"), dpi=200); plt.show()

In [ ]:
# 20. Compute cost of one-stage vs two-stage pipelines   [NEW: replaces the old timing cell]
# Serves: "report compute cost, since two-stage pipelines add latency".
# Latency (cell 11) is measured per sample, batch size 1, on PRELOADED arrays (no disk / PNG decoding), after warm-up, and the
# two-stage path is timed END-TO-END = CNN extractor forward + StandardScaler + head. Training cost of a two-stage pipeline =
# CNN training + head fitting. Hardware is recorded in outputs/config.json: quote it next to every number.
def load_latency():
    rows = []
    for f in sorted(glob.glob(os.path.join(RAW_DIR, "lat_b*.json"))):
        with open(f) as fh:
            rows += json.load(fh)
    return pd.DataFrame(rows)

LAT = load_latency()
if LAT.empty:
    print("No latency files found (they are written for SEEDS[0] only). Run cell 12 with the first seed.")
else:
    LAT["method"] = [mid(r.backbone, r.loss, r.head, r.tap) for r in LAT.itertuples()]
    agg = LAT.groupby("method").agg(params=("n_params", "first"), flops=("flops", "mean"), extractor_ms=("ext_ms", "mean"), head_ms=("head_ms", "mean"),
                                    total_ms_batch1=("total_ms", "mean"), total_ms_per_sample_batch256=("total_ms_b256", "mean")).reset_index()
    e2e_ms = {(r.backbone, r.loss): r.total_ms_batch1 for r in agg.assign(backbone=agg.method.str.split("|").str[0], loss=agg.method.str.split("|").str[1],
                                                                          head=agg.method.str.split("|").str[2]).query("head == 'e2e'").itertuples()}
    agg["backbone"] = agg.method.str.split("|").str[0]; agg["loss"] = agg.method.str.split("|").str[1]; agg["head"] = agg.method.str.split("|").str[2]
    agg["added_latency_vs_own_e2e_ms"] = [r.total_ms_batch1 - e2e_ms.get((r.backbone, r.loss), np.nan) if r.head not in ("e2e", "extractor") else 0.0 for r in agg.itertuples()]
    # training / fitting cost from the full result table (mean over seeds & benchmarks)
    tr = df.groupby("method").agg(backbone_train_s=("backbone_train_s", "mean"), head_fit_s=("fit_s", "mean"), pipeline_train_s=("pipeline_train_s", "mean")).reset_index()
    ref_e2e = agg.loc[agg.method == mid("ref", "focal", "e2e", "-"), "total_ms_batch1"]
    agg["added_vs_reference_baseline_ms"] = (agg.total_ms_batch1 - float(ref_e2e.iloc[0])) if len(ref_e2e) else np.nan
    COST = agg.merge(tr, on="method", how="left")
    COST.insert(0, "label", [pretty(m) if parse_mid(m)[2] != "extractor" else f"CNN extractor ({parse_mid(m)[3]}) [{BB_NAME.get(parse_mid(m)[0], parse_mid(m)[0])}, {parse_mid(m)[1]}]" for m in COST.method])
    COST = COST.drop(columns=["backbone", "loss", "head"])
    COST.to_csv(os.path.join(TAB_DIR, "compute_cost.csv"), index=False)
    pd.set_option("display.width", 250)
    print(COST.round(3).to_string(index=False))
    print("\nHardware:", ENV["gpus"] or "CPU only", "| TF", ENV["tensorflow"], "| FLOPs from the TF profiler (NaN if unavailable); heads have no FLOP estimate.")
    print("Paper wording: two-stage latency = 'extractor_ms + head_ms' (= total_ms_batch1); overhead = 'added_vs_reference_baseline_ms' "
          "(vs the reference CNN end-to-end, measured in the same cell-11b run). Latency: compiled tf.function graphs, batch size 1, "
          f"ICCAD-2 test clips, seed 0, {ENV['gpus'] or 'CPU'}, RF with n_jobs=1; 'total_ms_per_sample_batch256' is the batched figure.")

In [ ]:
# 21. Comparison with published ICCAD-12 results   [NEW]
# Serves: "compare against published ICCAD-12 results (hotspot recall, false alarms, F1), not only your own baseline".
# FILL `PUBLISHED` BY HAND from the cited papers (nothing is pre-filled on purpose: copy numbers only from the paper itself and note
# how each paper defines its metrics). Caveats to state in the paper:
#   * published "accuracy"/"Accu" for hotspot detection is usually RECALL = TP/(TP+FN), not balanced accuracy;
#   * "FA"/"false alarm" is the COUNT of non-hotspot clips flagged (our `fa` column = FP count on the official test set);
#   * check whether the paper reports each ICCAD-12 benchmark separately, a merged set, or a different clip-image dataset;
#   * F1 is rarely published: compute it from published TP/FP/FN if available, otherwise leave blank and say so.
PUBLISHED = pd.DataFrame(columns=["source", "method", "benchmark", "recall", "fa", "f1", "notes"])
# Example row format (replace with real values and a real citation):
# PUBLISHED.loc[len(PUBLISHED)] = ["[7]", "Method name", "ICCAD-2", 0.0, 0, None, "metric definition / protocol note"]

def ours_for_comparison():
    rows = []
    cands = [mid("ref", "focal", "e2e", "-")] + [mid(*MAIN_CONFIG, h, "late") for h in HEADS + ["ENS"]]
    for m in cands:
        if m not in set(df.method):
            continue
        for b in BENCHMARKS:
            d = df[(df.method == m) & (df.benchmark == b)]
            rows.append(dict(source="this work", method=pretty(m), benchmark=f"ICCAD-{b}", recall=d.recall.mean(), fa=d.fa.mean(), f1=d.f1.mean(),
                             notes=f"mean over {len(d)} seed(s); official test set; threshold chosen on validation"))
    return pd.DataFrame(rows)

OURS = ours_for_comparison()
COMPARE = pd.concat([PUBLISHED, OURS], ignore_index=True)
COMPARE.to_csv(os.path.join(TAB_DIR, "published_comparison.csv"), index=False)
print("Our numbers in the published-comparison format:")
print(OURS.round(3).to_string(index=False))
if PUBLISHED.empty:
    print("\nTODO: PUBLISHED is empty. Add rows from the literature before using this table in the paper.")

In [ ]:
# 22. Validation-only model selection, test set used once   [NEW]
# Serves: "select models, thresholds and hyper-parameters on a validation split only; use the test set once".
# DA-2 bolded the 'best' rows using TEST results. Here the headline method is chosen by VALIDATION balanced accuracy only, and only then
# is its (already frozen) test score read out. Both rules below are fixed before looking at test numbers.
#   Rule A (per benchmark & seed): among the methods in the main grid, pick the highest validation BA.
#   Rule B (global, per seed): pick the single method with the highest validation BA averaged over the 5 benchmarks.
# Caveat: the validation BA is optimistic (the threshold was tuned on it) and rests on only a handful of validation hotspots.
CAND = [m for m in df.method.unique() if parse_mid(m)[3] != "-" or parse_mid(m)[2] == "e2e"]
CAND = [m for m in CAND if parse_mid(m)[0] in ("ref", "proposed") and parse_mid(m)[1] in LOSSES]
sel_rows, glob_rows = [], []
for seed in SEEDS_PRESENT:
    for b in BENCHMARKS:
        d = df[(df.seed == seed) & (df.benchmark == b) & (df.method.isin(CAND))]
        if d.empty: continue
        best = d.loc[d.val_ba.idxmax()]
        sel_rows.append(dict(seed=seed, benchmark=b, selected=pretty(best.method), val_ba=best.val_ba, test_ba=best.balanced_accuracy, test_recall=best.recall, test_fa=best.fa))
    d = df[(df.seed == seed) & (df.method.isin(CAND))]
    g = d.groupby("method").val_ba.mean()
    if len(g):
        mbest = g.idxmax()
        for b in BENCHMARKS:
            r = d[(d.method == mbest) & (d.benchmark == b)]
            if len(r): glob_rows.append(dict(seed=seed, benchmark=b, selected=pretty(mbest), mean_val_ba=g.max(), test_ba=r.balanced_accuracy.iloc[0], test_recall=r.recall.iloc[0], test_fa=r.fa.iloc[0]))
SEL_A, SEL_B = pd.DataFrame(sel_rows), pd.DataFrame(glob_rows)
SEL_A.to_csv(os.path.join(TAB_DIR, "selection_rule_A_per_benchmark.csv"), index=False)
SEL_B.to_csv(os.path.join(TAB_DIR, "selection_rule_B_global.csv"), index=False)
print("Rule A: per-benchmark selection by validation BA -> test BA (mean ± std over seeds)")
print(SEL_A.groupby("benchmark").agg(test_ba_mean=("test_ba", "mean"), test_ba_std=("test_ba", "std"), test_recall=("test_recall", "mean"),
                                     test_fa=("test_fa", "mean"), most_often_selected=("selected", lambda s: s.value_counts().index[0])).round(3).to_string())
print("   benchmark-average per seed:", SEL_A.groupby("seed").test_ba.mean().round(3).tolist())
print("\nRule B: ONE method per seed (best mean validation BA) -> test BA")
print(SEL_B.groupby("seed").agg(selected=("selected", "first"), mean_val_ba=("mean_val_ba", "first"), test_ba_avg=("test_ba", "mean")).round(3).to_string())

In [ ]:
DETERMINISTIC = False   # same value as the original run (config.json says "deterministic": false)

In [ ]:
# 23. Export tables / figures / README for the repository and the report   [NEW]
# Serves: release-ready description of code and seeds.
README = f"""# Lithography hotspot detection: CNN features + classical heads (ICCAD-12) -- reproducibility notes

* Notebook: this file. Data: ICCAD-12 image clips (archive SHA-256: {DATASET_SHA256}).
* Seeds: {SEEDS} (completed here: {SEEDS_PRESENT}). One seed controls the stratified 85/15 train/validation split, weight
  initialisation, augmentation, shuffling and the RF / DT random_state. GPU kernels are not forced deterministic
  (DETERMINISTIC={DETERMINISTIC}), so identical seeds can differ slightly between runs; across-seed variance is reported.
* Protocol: each benchmark is treated independently (never merged). Model hyper-parameters (SVM C, DT depth/leaf), decision
  thresholds and early stopping use the VALIDATION split only; the official test set is scored once per trained model.
* Shared CNN training recipe for BOTH backbones: {json.dumps(TRAIN_CFG)}; focal loss {FOCAL}; weighted BCE caps the positive class
  weight at {CLASS_WEIGHT_CAP}.
* Grid: backbones {BACKBONES} x losses {LOSSES} x heads {HEADS + ['ENS']}; early/late/combined taps only for {MAIN_CONFIG}.
* Environment: {json.dumps(ENV)}
* Outputs: outputs/raw (per-run scores .npz + metric rows .json), outputs/tables (CSV), outputs/figures (PNG), outputs/config.json.
* Re-run: set QUICK_PILOT=False, run cells 1-12 once per seed (RUN_SEEDS=[k]), then cells 14-23.
* Compute: wall-clock for each run is stored in the rows (train_s, fit_s); latency/FLOPs in raw/lat_*.json (first seed only).
"""
with open(os.path.join(OUT_DIR, "README_reproducibility.md"), "w") as f:
    f.write(README)
print(README)
zip_base = os.path.join(BASE, "da3_tables_figures")
shutil.make_archive(zip_base, "zip", OUT_DIR, "tables")      # tables only; add 'figures' / 'raw' separately if wanted
shutil.make_archive(zip_base + "_figures", "zip", OUT_DIR, "figures")
print("Wrote:", zip_base + ".zip", "and", zip_base + "_figures.zip")
shutil.make_archive(os.path.join(BASE, "da3_raw"), "zip", OUT_DIR, "raw")
shutil.rmtree(os.path.join(BASE, "iccad12_dataset"), ignore_errors=True)
if os.path.exists(ARCHIVE): os.remove(ARCHIVE)